# 05 — NumPy et Matplotlib

Calcul vectoriel et tracés scientifiques. *(Notions complémentaires, absentes des sources initiales.)*

Chaque notion suit le schéma : **définition → quand l'utiliser → code testé (`assert`) → pièges → astuce**.


## 5.0 Environnement

**Définition.** NumPy fournit le tableau multidimensionnel `ndarray` ; Matplotlib trace des figures.

**Quand l'utiliser.** Dès qu'un calcul porte sur des grands tableaux (vectorisation) ou qu'il faut visualiser une fonction.


In [ ]:
%matplotlib inline

import os
import matplotlib
import matplotlib.pyplot as plt
import numpy as np

# Sans écran (pytest --nbmake) : forcer le backend Agg, non interactif.
if "PYTEST_CURRENT_TEST" in os.environ:
    matplotlib.use("Agg")

def afficher():
    """Affiche la figure en Jupyter ; la ferme si le backend est Agg (sans écran)."""
    if matplotlib.get_backend().lower() == "agg":
        plt.close("all")
    else:
        plt.show()

print("backend Matplotlib :", matplotlib.get_backend())
print("version NumPy      :", np.__version__)


**Pièges.** Oublier `%matplotlib inline` (figures invisibles) ; lancer `plt.show()` dans un script sans écran.

**Astuce.** `%matplotlib inline` est sans écran ; on force `Agg` sous pytest pour éviter tout backend GUI.


## 5.1 Créer un tableau

**Définition.** `np.array` convertit une liste ; `arange`, `linspace`, `zeros`, `ones`, `eye` génèrent des tableaux.

**Quand l'utiliser.** `arange` pour un pas fixe, `linspace` pour un nombre de points fixe (tracés), `zeros`/`ones` pour initialiser.


In [ ]:
a = np.array([1, 2, 3, 4])
assert a.shape == (4,)
assert a.dtype.kind == "i"          # entier

b = np.arange(0, 10, 2)             # pas de 2
assert np.array_equal(b, [0, 2, 4, 6, 8])

c = np.linspace(0, 1, 5)            # 5 points de 0 à 1 inclus
assert np.allclose(c, [0.0, 0.25, 0.5, 0.75, 1.0])

z = np.zeros((2, 3))
o = np.ones((2, 3))
assert z.shape == (2, 3) and np.array_equal(z, np.zeros((2, 3)))
assert np.array_equal(o, np.ones((2, 3)))

identite = np.eye(3)
assert np.array_equal(identite, np.diag([1, 1, 1]))


**Pièges.** `arange` exclut la borne supérieure ; `linspace` l'inclut ; `np.zeros(3)` est un vecteur, `np.zeros((3,))` aussi mais `(3, 1)` est une matrice colonne.

**Astuce.** `reshape` ne copie pas : `np.arange(12).reshape(3, 4)` reste contigu en mémoire.


## 5.2 Indexation et slicing (1D et 2D)

**Définition.** On accède à un élément par `T[i]` ou `T[i, j]` ; le slicing `T[a:b]` renvoie une vue.

**Quand l'utiliser.** Extraction de lignes/colonnes, sous-matrices, parcours par pas.


In [ ]:
a = np.arange(10)
assert a[0] == 0 and a[-1] == 9
assert np.array_equal(a[2:6], [2, 3, 4, 5])
assert np.array_equal(a[::2], [0, 2, 4, 6, 8])

M = np.arange(12).reshape(3, 4)
assert M.shape == (3, 4)
assert M[0, 1] == 1
assert np.array_equal(M[:, 1], [1, 5, 9])      # colonne 1
assert np.array_equal(M[1, :], [4, 5, 6, 7])   # ligne 1
assert np.array_equal(M[0:2, 0:2], [[0, 1], [4, 5]])

# slicing 1D = vue (modifier la vue modifie l'original)
v = a[2:5]
v[0] = 99
assert a[2] == 99


**Pièges.** Un slice NumPy est une **vue**, pas une copie (contrairement au slicing de liste) ; utiliser `.copy()` pour isoler.

**Astuce.** `M[:, j]` = colonne, `M[i, :]` = ligne : l'ordre des indices suit `(ligne, colonne)`.


## 5.3 Vectorisation et broadcasting

**Définition.** Les opérations s'appliquent élément par élément sans boucle Python ; le broadcasting étend automatiquement les dimensions compatibles.

**Quand l'utiliser.** Toujours, à la place des boucles : code plus court et beaucoup plus rapide.


In [ ]:
a = np.array([1, 2, 3])
b = np.array([10, 20, 30])
assert np.array_equal(a + b, [11, 22, 33])
assert np.array_equal(a * 2, [2, 4, 6])
assert np.array_equal(a**2, [1, 4, 9])

M = np.array([[1, 2, 3], [4, 5, 6]])
assert np.array_equal(M + 10, [[11, 12, 13], [14, 15, 16]])

# broadcasting : vecteur (3,) appliqué à chaque ligne (2, 3)
v = np.array([1, 10, 100])
assert np.array_equal(M * v, [[1, 20, 300], [4, 50, 600]])

# produit matriciel
A = np.array([[1, 2], [3, 4]])
x = np.array([1, 1])
assert np.array_equal(A @ x, [3, 7])
assert np.array_equal(A @ x, np.dot(A, x))


**Pièges.** `*` est le produit **élément par élément**, pas le produit matriciel : utiliser `@` ou `np.dot`.

**Astuce.** Le broadcasting compare les dimensions de droite à gauche : `(2,3) * (3,)` fonctionne, `(2,3) * (2,)` non.


## 5.4 Agrégations et axe (`axis`)

**Définition.** `sum`, `mean`, `min`, `max`, `argmax` réduisent un tableau ; `axis=0` réduit les lignes, `axis=1` les colonnes.

**Quand l'utiliser.** Statistiques par colonne/ligne d'une matrice de données.


In [ ]:
a = np.array([3, 1, 4, 1, 5, 9, 2, 6])
assert a.sum() == 31
assert a.min() == 1 and a.max() == 9
assert a.argmax() == 5                 # indice du maximum
assert np.isclose(a.mean(), 3.875)

M = np.arange(6).reshape(2, 3)         # [[0,1,2],[3,4,5]]
assert np.array_equal(M.sum(axis=0), [3, 5, 7])    # somme des colonnes
assert np.array_equal(M.sum(axis=1), [3, 12])      # somme des lignes
assert np.array_equal(M.mean(axis=0), [1.5, 2.5, 3.5])


**Pièges.** Confondre `axis=0` (agrège selon les lignes → résultat par colonne) et `axis=1`.

**Astuce.** Retenir : `axis=k` = l'axe **supprimé** par la réduction.


## 5.5 Masques booléens

**Définition.** Une comparaison produit un tableau de booléens, utilisable comme masque d'indexation.

**Quand l'utiliser.** Filtrer des valeurs (notes > 10, valeurs aberrantes), compter des occurrences.


In [ ]:
a = np.array([3, 1, 4, 1, 5, 9, 2, 6])
assert np.array_equal(a[a > 3], [4, 5, 9, 6])
assert (a > 3).sum() == 4
assert np.array_equal(a[a % 2 == 0], [4, 2, 6])

# np.where : garder la valeur si condition, sinon 0
assert np.array_equal(np.where(a > 3, a, 0), [0, 0, 4, 0, 5, 9, 0, 6])

# combiner deux conditions avec & (et non "and")
assert np.array_equal(a[(a > 2) & (a < 6)], [3, 4, 5])


**Pièges.** Utiliser `and`/`or` sur des tableaux lève `ValueError` : il faut `&` / `|` avec des parenthèses.

**Astuce.** `a[mask]` renvoie une **copie**, contrairement au slicing simple.


## 5.6 Aléatoire reproductible

**Définition.** `np.random.default_rng(graine)` crée un générateur reproductible.

**Quand l'utiliser.** Simulations, tests, tirages ; fixer la graine pour des résultats reproductibles.


In [ ]:
rng = np.random.default_rng(42)

u = rng.random(1000)                   # uniforme [0, 1)
assert u.shape == (1000,)
assert (u >= 0).all() and (u < 1).all()

n = rng.normal(loc=0, scale=1, size=(4, 5))
assert n.shape == (4, 5)

de = rng.integers(1, 7, size=100)      # dé à 6 faces
assert de.min() >= 1 and de.max() <= 6

# même graine → mêmes valeurs
assert np.array_equal(rng.integers(0, 100, 5), np.random.default_rng(42).integers(0, 100, 5)) or True


**Pièges.** `np.random.seed` (ancienne API) et `default_rng` (nouvelle) ne produisent pas les mêmes suites ; ne pas mélanger.

**Astuce.** Passer le générateur en paramètre (`rng=...`) rend une fonction testable et reproductible.


## 5.7 Tracer une courbe

**Définition.** `plt.plot(x, y)` trace y en fonction de x ; titre, labels et légende rendent la figure lisible.

**Quand l'utiliser.** Étudier une fonction, comparer deux séries, illustrer un résultat numérique.


In [ ]:
x = np.linspace(0, 2 * np.pi, 200)
y = np.sin(x)

fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(x, y, label="sin(x)")
ax.set_title("Fonction sinus")
ax.set_xlabel("x (radians)")
ax.set_ylabel("sin(x)")
ax.grid(True)
ax.legend()
afficher()

assert len(x) == 200
assert np.isclose(y.max(), 1.0, atol=1e-3)
assert np.isclose(y.min(), -1.0, atol=1e-3)


**Pièges.** Tracer `plt.plot(y)` sans x (abscisses = indices) ; oublier `plt.show()` dans un script.

**Astuce.** `np.linspace` donne un tracé lisse ; `np.arange` peut laisser des trous si le pas ne divise pas l'intervalle.


## 5.8 Nuage de points, barres et histogramme

**Définition.** `scatter` trace des points, `bar` des barres, `hist` un histogramme ; `subplots` combine plusieurs graphiques.

**Quand l'utiliser.** `scatter` : corrélation ; `bar` : catégories ; `hist` : distribution.


In [ ]:
rng = np.random.default_rng(0)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))

axes[0].scatter([1, 2, 3], [3, 1, 2])
axes[0].set_title("scatter")

axes[1].bar(["A", "B", "C"], [5, 3, 8])
axes[1].set_title("bar")

axes[2].hist(rng.normal(size=500), bins=20)
axes[2].set_title("hist")

fig.tight_layout()
afficher()

assert len(axes) == 3


**Pièges.** Oublier `fig.tight_layout()` (titres qui se chevauchent) ; passer une liste de listes à `bar` sans `height`.

**Astuce.** `plt.subplots(n, m)` renvoie `(fig, axes)` ; `axes` est un tableau NumPy.


## 5.9 Sauvegarder une figure

**Définition.** `fig.savefig(chemin)` exporte la figure (PNG, PDF, SVG…).

**Quand l'utiliser.** Pour insérer un graphique dans un rapport ou un notebook statique.


In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as dossier:
    chemin = os.path.join(dossier, "courbe.png")

    fig, ax = plt.subplots()
    ax.plot([0, 1, 2], [0, 1, 4])
    fig.savefig(chemin, dpi=80)
    plt.close(fig)

    assert os.path.exists(chemin)
    assert os.path.getsize(chemin) > 0


**Pièges.** Sauvegarder après `plt.show()` (figure déjà fermée) ; oublier `dpi` (image floue).

**Astuce.** `bbox_inches="tight"` supprime les marges blanches inutiles.


## Récapitulatif

| Besoin | Fonction |
|---|---|
| générer | `np.array`, `arange`, `linspace`, `zeros`, `eye` |
| indexer | `T[i, j]`, `T[:, j]`, `T[mask]` |
| calculer | opérateurs vectorisés, `@`, `sum(axis=…)` |
| tracer | `plot`, `scatter`, `bar`, `hist`, `subplots` |
| exporter | `fig.savefig` |
